# Create and Manage Prompt versions

In [1]:
#Defining 2 prompts
prompt_v1 = """
You are a helpful customer support assistant.
Answer customer questions.
"""

prompt_v2 = """
You are a professional customer support assistant.

Use simple English.
Be polite and concise.
Answer using only the provided company information.
If the information is unavailable, say you do not know.
Never invent company policies.
"""

In [2]:
#Creating Prompt Reqgistry
prompt_registry = {
    "customer_support": {
        "v1": prompt_v1,
        "v2": prompt_v2
    }
}

print(prompt_registry)

{'customer_support': {'v1': '\nYou are a helpful customer support assistant.\nAnswer customer questions.\n', 'v2': '\nYou are a professional customer support assistant.\n\nUse simple English.\nBe polite and concise.\nAnswer using only the provided company information.\nIf the information is unavailable, say you do not know.\nNever invent company policies.\n'}}


In [3]:
#Retrieving a Specific version
def get_prompt(prompt_name, version):

    if prompt_name not in prompt_registry:
        raise ValueError("Prompt does not exist.")

    if version not in prompt_registry[prompt_name]:
        raise ValueError("Prompt version does not exist.")

    return prompt_registry[prompt_name][version]

In [4]:
selected_prompt = get_prompt(
    "customer_support",
    "v2"
)

print(selected_prompt)


You are a professional customer support assistant.

Use simple English.
Be polite and concise.
Answer using only the provided company information.
If the information is unavailable, say you do not know.
Never invent company policies.



In [5]:
#Saving Prompt versions to JSON
import json
from pathlib import Path

PROMPT_DIR = Path("prompts")

PROMPT_DIR.mkdir(exist_ok=True)

PROMPT_FILE = PROMPT_DIR / "prompt_versions.json"

with open(PROMPT_FILE, "w", encoding="utf-8") as file:

    json.dump(
        prompt_registry,
        file,
        indent=4
    )

print("Prompt versions saved.")

Prompt versions saved.


In [6]:
#Loading prompt versions
def load_prompt_registry():

    with open(PROMPT_FILE, "r", encoding="utf-8") as file:

        return json.load(file)


loaded_registry = load_prompt_registry()

print(loaded_registry)

{'customer_support': {'v1': '\nYou are a helpful customer support assistant.\nAnswer customer questions.\n', 'v2': '\nYou are a professional customer support assistant.\n\nUse simple English.\nBe polite and concise.\nAnswer using only the provided company information.\nIf the information is unavailable, say you do not know.\nNever invent company policies.\n'}}


In [7]:
#Creating Test Cases
test_cases = [
    {
        "id": "case_001",
        "question": "What is your refund period?",
        "context": "Customers can request a refund within 30 days of purchase.",
        "expected_keywords": ["30 days", "refund"]
    },
    {
        "id": "case_002",
        "question": "Do you offer free delivery?",
        "context": "The company offers standard delivery. No information about free delivery is available.",
        "expected_keywords": ["not available", "information"]
    },
    {
        "id": "case_003",
        "question": "How can I contact customer support?",
        "context": "Customers can contact support at support@example.com.",
        "expected_keywords": ["support@example.com"]
    }
]

In [8]:
#Saving the evaluation dataset
EXPERIMENT_DIR = Path("experiments")

EXPERIMENT_DIR.mkdir(exist_ok=True)

DATASET_FILE = EXPERIMENT_DIR / "test_cases.json"

with open(DATASET_FILE, "w", encoding="utf-8") as file:

    json.dump(
        test_cases,
        file,
        indent=4
    )

In [9]:
#Build an evaluation function.For this exercise, we'll use a simple keyword-coverage score.
#It checks how many expected keywords appear in the answer.
#Implement keyword coverage

def keyword_coverage(answer, expected_keywords):

    answer = answer.lower()

    if not expected_keywords:
        return 0.0

    matched_keywords = []

    for keyword in expected_keywords:

        if keyword.lower() in answer:
            matched_keywords.append(keyword)

    score = len(matched_keywords) / len(expected_keywords)

    return score

In [10]:
answer = "Customers can request a refund within 30 days."

keywords = ["refund", "30 days"]

score = keyword_coverage(answer, keywords)

print("Score:", score)

Score: 1.0


# Running a real prompt experiment

In [11]:
%pip install --upgrade openai python-dotenv

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ---- --------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 1.15.30 requires openai<3,>=2.30.0, but you have openai 3.49.0 which is incompatible.
guardrails-ai 0.11.0 requires openai<3.0.0,>=2.0.0, but you have openai 3.49.0 which is incompatible.
instructor 1.15.4 requires jiter<0.15,>=0.6.1, but you have jiter 0.18.0 which is incompatible.
instructor 1.15.4 requires openai<3.0.0,>=2.0.0, but you have openai 3.49.0 which is incompatible.


In [12]:
import os
import time
import uuid

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY was not found.")

client = OpenAI(api_key=api_key)

In [13]:
#Creating the experiment function
def run_single_test(prompt, test_case):

    user_input = f"""
    Customer question:
    {test_case["question"]}

    Company information:
    {test_case["context"]}
    """

    start_time = time.perf_counter()

    response = client.responses.create(
        model="gpt-4.1-mini",
        instructions=prompt,
        input=user_input,
        max_output_tokens=200
    )

    latency_ms = (
        time.perf_counter() - start_time
    ) * 1000

    answer = response.output_text

    usage = response.usage

    input_tokens = usage.input_tokens if usage else None
    output_tokens = usage.output_tokens if usage else None

    score = keyword_coverage(
        answer,
        test_case["expected_keywords"]
    )

    return {
        "test_case_id": test_case["id"],
        "answer": answer,
        "keyword_coverage": round(score, 3),
        "latency_ms": round(latency_ms, 2),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens
    }

In [14]:
#Running version 1
prompt = get_prompt(
    "customer_support",
    "v1"
)

result_v1 = run_single_test(
    prompt,
    test_cases[0]
)

print(json.dumps(result_v1, indent=4))

{
    "test_case_id": "case_001",
    "answer": "Our refund period is 30 days from the date of purchase. If you request a refund within this time frame, we will be happy to assist you.",
    "keyword_coverage": 1.0,
    "latency_ms": 6079.51,
    "input_tokens": 54,
    "output_tokens": 32
}


In [15]:
#Running Version2 
prompt = get_prompt(
    "customer_support",
    "v2"
)

result_v2 = run_single_test(
    prompt,
    test_cases[0]
)

print(json.dumps(result_v2, indent=4))

{
    "test_case_id": "case_001",
    "answer": "You can request a refund within 30 days of purchase.",
    "keyword_coverage": 1.0,
    "latency_ms": 4685.27,
    "input_tokens": 84,
    "output_tokens": 13
}


In [16]:
#Creating an experiment runner
def run_experiment(prompt_name, version, test_cases):

    prompt = get_prompt(prompt_name, version)

    experiment_id = str(uuid.uuid4())

    results = []

    for test_case in test_cases:

        try:

            result = run_single_test(
                prompt,
                test_case
            )

            result["status"] = "success"

        except Exception as error:

            result = {
                "test_case_id": test_case["id"],
                "status": "error",
                "error_type": type(error).__name__
            }

        results.append(result)

    experiment = {
        "experiment_id": experiment_id,
        "prompt_name": prompt_name,
        "prompt_version": version,
        "model": "gpt-4.1-mini",
        "dataset": DATASET_FILE.name,
        "created_at": time.time(),
        "results": results
    }

    return experiment

In [17]:
#Running both experiments
experiment_v1 = run_experiment(
    "customer_support",
    "v1",
    test_cases
)

experiment_v2 = run_experiment(
    "customer_support",
    "v2",
    test_cases
)

In [18]:
#Saving experiment results
EXPERIMENT_RESULTS_FILE = (
    EXPERIMENT_DIR / "experiment_results.json"
)

all_experiments = [
    experiment_v1,
    experiment_v2
]

with open(
    EXPERIMENT_RESULTS_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        all_experiments,
        file,
        indent=4
    )

print("Experiment results saved.")

Experiment results saved.


In [19]:
#Creating a summary fucntion
def summarize_experiment(experiment):

    successful_results = [
        result
        for result in experiment["results"]
        if result["status"] == "success"
    ]

    if not successful_results:
        return {
            "version": experiment["prompt_version"],
            "successful_tests": 0
        }

    average_score = sum(
        result["keyword_coverage"]
        for result in successful_results
    ) / len(successful_results)

    average_latency = sum(
        result["latency_ms"]
        for result in successful_results
    ) / len(successful_results)

    total_input_tokens = sum(
        result["input_tokens"] or 0
        for result in successful_results
    )

    total_output_tokens = sum(
        result["output_tokens"] or 0
        for result in successful_results
    )

    return {
        "version": experiment["prompt_version"],
        "successful_tests": len(successful_results),
        "average_keyword_coverage": round(average_score, 3),
        "average_latency_ms": round(average_latency, 2),
        "total_input_tokens": total_input_tokens,
        "total_output_tokens": total_output_tokens
    }

In [20]:
#Comparing results
summary_v1 = summarize_experiment(experiment_v1)

summary_v2 = summarize_experiment(experiment_v2)

print("VERSION 1")
print(json.dumps(summary_v1, indent=4))

print("\nVERSION 2")
print(json.dumps(summary_v2, indent=4))

VERSION 1
{
    "version": "v1",
    "successful_tests": 3,
    "average_keyword_coverage": 0.833,
    "average_latency_ms": 1780.71,
    "total_input_tokens": 162,
    "total_output_tokens": 88
}

VERSION 2
{
    "version": "v2",
    "successful_tests": 3,
    "average_keyword_coverage": 0.833,
    "average_latency_ms": 1291.78,
    "total_input_tokens": 252,
    "total_output_tokens": 48
}


In [21]:
#Saving the exact promtp shot
experiment_v1["prompt_snapshot"] = get_prompt(
    "customer_support",
    "v1"
)

experiment_v2["prompt_snapshot"] = get_prompt(
    "customer_support",
    "v2"
)